# Carrier Isolation Study: Co-channel vs Isolated gNBs

**Research question:** Does co-channel interference corrupt the useful-PRB load metric used in the Jain fairness reward?

## Setup
- Scenario: `jain_balance_controllable` — 6 eMBB UEs at equidistant midpoints (x=±135 m), all initially on gNB-1 (center, x=0)
- One action: full negative bias (-1.0) → 6 handovers fire to outer gNBs (3 each)
- Two configurations run in parallel:
  - **Co-channel**: all gNBs share `carrier_id=0` → mutual interference via `_same_carrier_interferers`
  - **Isolated**: gNB-0/1/2 → `carrier_id=0/1/2` → empty interferer list → SINR = SNR (noise-limited)

## Expected findings
After HO, UEs at (x=±135 m) are **equidistant** (135 m) from:
- Their new serving outer gNB
- **gNB-1 (center), which transmits on the same carrier even when idle**

Co-channel: Signal ≈ Interference → SINR ≈ 0 dB → catastrophic spectral efficiency → scheduler piles all 100 PRBs → `gnb_total_useful_load = 1.0` regardless of demand.  
Isolated: No interference → high SINR → ~15 PRBs/UE as designed → `gnb_total_useful_load ≈ 0.45`.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

_base = CENTER_GAP_GNB_CONFIGS['medium_270m']

# Co-channel: all share carrier_id=0 (current training default)
COCHANNEL_CONFIGS = _base

# Isolated: unique carrier per gNB — _same_carrier_interferers will be empty
ISOLATED_CONFIGS  = tuple({**g, 'carrier_id': g['id']} for g in _base)

GNB_POSITIONS = {g['id']: (g['x'], g['y']) for g in _base}
GNB_COLORS    = ['#1565C0', '#6A1B9A', '#1B5E20']
SCENARIO  = 'jain_balance_controllable'
N_SETTLE  = 12   # upper steps observed after HO

print('Co-channel carrier IDs :', [g['carrier_id'] for g in COCHANNEL_CONFIGS])
print('Isolated  carrier IDs  :', [g['carrier_id'] for g in ISOLATED_CONFIGS])
print('gNB positions          :', GNB_POSITIONS)

In [ ]:
def make_env(gnb_configs, warmup_steps=4):
    return GlobalPPO3GNBEnv(
        seed=0,
        scenario_mode='curriculum',
        training_scenarios=SCENARIO,
        gnb_configs=gnb_configs,
        global_steps_per_episode=24,
        local_steps_per_global=10,
        radio_substeps=20,
        warmup_steps=warmup_steps,
        safe_admission_enabled=False,
        terminal_reward_only=False,
        post_handover_settle_steps=4,
    )

def gnb_loads(env):
    loads = env.base_env.get_slice_loads()
    return np.clip(
        np.array([sum(loads.get((g, s), 0.0) for s in SLICE_TYPES) for g in range(3)]),
        0.0, 1.0,
    )

def jain(v):
    v = np.asarray(v, dtype=float)
    s2 = float(np.sum(v ** 2))
    return float(np.sum(v) ** 2) / (3 * s2) if s2 > 0 else 1.0

def get_ue_metrics(env):
    """SINR and calibration state for every connected UE."""
    env.base_env._invalidate_metric_caches()
    rows = []
    for ue in env.base_env.get_all_ues():
        if not ue.connected or ue.serving_gnb is None:
            continue
        gnb = env.base_env._gnb_by_id[int(ue.serving_gnb)]
        m   = env.base_env._compute_link_metrics(gnb, ue)
        rows.append({
            'ue_id'    : int(ue.id),
            'gnb'      : int(ue.serving_gnb),
            'x'        : float(ue.x),
            'sinr_db'  : float(m['sinr_db']),
            'interf_db': float(m.get('interference_dbm', -100.0)),
            'bit_rate' : float(getattr(getattr(ue, 'traffic_source', None), 'bit_rate', 0.0)),
            'demand_prbs': int(getattr(ue, 'upper_demand_prbs', 0)),
        })
    return rows

print('Helpers ready')

---
## Run experiment for both configurations

In [ ]:
def run_experiment(gnb_configs, label):
    env = make_env(gnb_configs, warmup_steps=4)
    env.reset()

    zero = np.zeros(env.action_space.shape, dtype=np.float32)
    neg  = np.full(env.action_space.shape, -1.0, dtype=np.float32)

    records       = []
    sinr_snapshots = {}

    def snapshot(tag, step_num):
        gl  = gnb_loads(env)
        ues = get_ue_metrics(env)
        avg_sinr    = float(np.mean([u['sinr_db']  for u in ues])) if ues else 0.0
        avg_interf  = float(np.mean([u['interf_db'] for u in ues])) if ues else -100.0
        avg_bitrate = float(np.mean([u['bit_rate']  for u in ues])) if ues else 0.0
        n_per_gnb   = {g: sum(1 for u in ues if u['gnb'] == g) for g in range(3)}
        records.append({
            'step'     : step_num,
            'tag'      : tag,
            'gnb0'     : float(gl[0]),
            'gnb1'     : float(gl[1]),
            'gnb2'     : float(gl[2]),
            'jain'     : jain(gl),
            'avg_sinr' : avg_sinr,
            'avg_interf': avg_interf,
            'avg_bitrate_mbps': avg_bitrate / 1e6,
            'n_gnb0'   : n_per_gnb[0],
            'n_gnb1'   : n_per_gnb[1],
            'n_gnb2'   : n_per_gnb[2],
        })
        return ues

    # step 0: one extra zero step after warmup → before HO
    env.step(zero)
    sinr_snapshots['before'] = snapshot('before', 0)

    # step 1: negative bias → handovers fire
    _, rew, _, _, info = env.step(neg)
    ho_count = int(info.get('handover_count', 0))
    sinr_snapshots['after_HO'] = snapshot('HO', 1)

    # settle steps
    for i in range(N_SETTLE):
        env.step(zero)
        snapshot(f'+{i+1}', i + 2)

    env.close()
    df = pd.DataFrame(records)
    df['config'] = label
    return df, sinr_snapshots, ho_count

print('Running co-channel ...')
df_cc,  snaps_cc,  ho_cc  = run_experiment(COCHANNEL_CONFIGS, 'co-channel')
print(f'  HOs: {ho_cc}')

print('Running isolated ...')
df_iso, snaps_iso, ho_iso = run_experiment(ISOLATED_CONFIGS,  'isolated')
print(f'  HOs: {ho_iso}')

print('\n--- Co-channel summary ---')
print(df_cc[['tag','gnb0','gnb1','gnb2','jain','avg_sinr','avg_bitrate_mbps']].to_string(index=False))
print('\n--- Isolated summary ---')
print(df_iso[['tag','gnb0','gnb1','gnb2','jain','avg_sinr','avg_bitrate_mbps']].to_string(index=False))

---
## 1 — Per-UE SINR: before vs after HO

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

def sinr_bar(ax, ue_rows, title):
    if not ue_rows:
        ax.text(0.5, 0.5, 'No UEs', ha='center', va='center', transform=ax.transAxes)
        return
    ids    = [u['ue_id']   for u in ue_rows]
    sinrs  = [u['sinr_db'] for u in ue_rows]
    gnbs   = [u['gnb']     for u in ue_rows]
    colors = [GNB_COLORS[g] for g in gnbs]
    bars   = ax.bar(range(len(ids)), sinrs, color=colors, edgecolor='white', alpha=0.9)
    for bar, s, g in zip(bars, sinrs, gnbs):
        ax.text(bar.get_x() + bar.get_width()/2,
                max(bar.get_height(), 0) + 0.5,
                f'{s:.1f} dB', ha='center', fontsize=8)
    ax.axhline(0,  color='crimson', lw=1.5, ls='--', alpha=0.7, label='0 dB (SINR collapse)')
    ax.axhline(10, color='orange',  lw=1,   ls=':',  alpha=0.7, label='10 dB (moderate)')
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels([f'UE-{i}\n(gNB-{g})' for i, g in zip(ids, gnbs)], fontsize=8)
    ax.set_ylabel('SINR (dB)')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(axis='y', alpha=0.3)
    lo = min(sinrs + [0]) - 5
    hi = max(sinrs) + 10
    ax.set_ylim(lo, hi)

sinr_bar(axes[0,0], snaps_cc['before'],   'Co-channel: BEFORE HO\n(all 6 UEs on gNB-1, outer gNBs idle)')
sinr_bar(axes[0,1], snaps_cc['after_HO'], 'Co-channel: AFTER HO\n(gNB-1 idle → still interferes at equal dist)')
sinr_bar(axes[1,0], snaps_iso['before'],  'Isolated: BEFORE HO\n(all 6 UEs on gNB-1)')
sinr_bar(axes[1,1], snaps_iso['after_HO'],'Isolated: AFTER HO\n(no cross-carrier interference)')

legend_els = [Patch(color=c, label=f'gNB-{i} (serving)') for i, c in enumerate(GNB_COLORS)]
fig.legend(handles=legend_els, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.02), framealpha=0.9)
fig.suptitle(
    'Per-UE SINR before and after handover\n'
    'Co-channel: gNB-1 (idle, x=0) is equidistant from outer UEs at x=±135 m → interference = signal → SINR ≈ 0 dB\n'
    'Isolated: gNB-1 on different carrier → no interference → SINR stays high',
    fontsize=10, fontweight='bold',
)
plt.tight_layout()
plt.show()

---
## 2 — PRB load and Jain convergence after HO

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
ax_outer, ax_center, ax_jain = axes

styles = {
    'co-channel': dict(linestyle='-',  marker='o', ms=5, lw=2.5),
    'isolated'  : dict(linestyle='--', marker='s', ms=5, lw=2.5),
}
colors_cfg = {'co-channel': '#E53935', 'isolated': '#1E88E5'}

x_cc  = df_cc['step'].values
x_iso = df_iso['step'].values
tags  = df_cc['tag'].tolist()

# Outer gNB average (gNB-0 and gNB-2 are symmetric)
for df, cfg_label in [(df_cc, 'co-channel'), (df_iso, 'isolated')]:
    outer_avg = (df['gnb0'] + df['gnb2']) / 2
    ax_outer.plot(df['step'], outer_avg,
                  color=colors_cfg[cfg_label], label=cfg_label, **styles[cfg_label])
    ax_center.plot(df['step'], df['gnb1'],
                   color=colors_cfg[cfg_label], label=cfg_label, **styles[cfg_label])
    ax_jain.plot(df['step'], df['jain'],
                 color=colors_cfg[cfg_label], label=cfg_label, **styles[cfg_label])

ax_outer.axhline(0.45, color='forestgreen', lw=1.5, ls=':', alpha=0.8,
                 label='expected 0.45  (3 UEs × 15 PRBs / 100)')
ax_outer.axhline(0.80, color='gray', lw=1, ls=':', alpha=0.5, label='overload limit')
ax_jain.axhline(0.667, color='forestgreen', lw=1.5, ls=':', alpha=0.8,
                label='expected Jain 0.667  (2 equal outer gNBs)')
ax_jain.axhline(1.000, color='gold', lw=1, ls=':', alpha=0.5, label='perfect Jain 1.0')

for ax in axes:
    ax.axvline(1, color='black', lw=2, ls='--', alpha=0.4)
    ax.text(1.1, 0.05, 'HO\nevent', transform=ax.get_xaxis_transform(),
            fontsize=8, color='black', alpha=0.6)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

ax_outer.set_ylabel('Outer gNB avg useful PRB load', fontsize=9)
ax_outer.set_title('Average gnb_total_useful_load  for gNB-0 and gNB-2 (outer, each receives 3 UEs)', fontsize=9)
ax_center.set_ylabel('gNB-1 (center) useful PRB load', fontsize=9)
ax_center.set_title('gNB-1 load  — should drop to 0 after all UEs leave', fontsize=9)
ax_jain.set_ylabel('Jain fairness index', fontsize=9)
ax_jain.set_title('Jain reward signal quality  — co-channel stays wrong; isolated converges quickly', fontsize=9)
ax_jain.set_ylim(0.25, 1.05)

ax_jain.set_xticks(x_cc)
ax_jain.set_xticklabels(tags, rotation=30, ha='right', fontsize=8)
ax_jain.set_xlabel('Upper step (step 1 = HO, step 2+ = settle)')

fig.suptitle(
    'PRB load and Jain reward convergence after handover\n'
    'Co-channel: load stays at 1.0 (SINR ≈ 0 dB → scheduler saturated → calibration takes 24+ steps)\n'
    'Isolated: load converges to expected 0.45 within a few steps → clean training signal',
    fontsize=10, fontweight='bold',
)
plt.tight_layout()
plt.show()

---
## 3 — Calibration convergence (mean offered bit_rate per UE)

After HO at low SINR, the calibration loop repeatedly applies:
```
correction  = clip(desired_prbs / achieved_prbs, 0.25, 4.0)
new_rate    = old_rate × [(1-α) + α × correction]
```
With `α=0.5` and `correction=0.25` (worst-case clamp), each step multiplies rate by `0.625`.  
From 12 Mbps → needed rate for 15 PRBs at 0 dB SINR ≈ 0.74 Mbps: requires **10.6 steps** to converge.
Even then, 3 UEs × 33 PRBs each = 99 PRBs → load still ≈ 1.0 at 0 dB SINR.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_rate, ax_sinr = axes

for df, cfg_label in [(df_cc, 'co-channel'), (df_iso, 'isolated')]:
    ax_rate.plot(df['step'], df['avg_bitrate_mbps'],
                 color=colors_cfg[cfg_label], label=cfg_label, **styles[cfg_label])
    ax_sinr.plot(df['step'], df['avg_sinr'],
                 color=colors_cfg[cfg_label], label=cfg_label, **styles[cfg_label])

for ax in (ax_rate, ax_sinr):
    ax.axvline(1, color='black', lw=2, ls='--', alpha=0.4)
    ax.set_xticks(x_cc)
    ax.set_xticklabels(tags, rotation=30, ha='right', fontsize=7)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)

ax_rate.set_ylabel('Mean offered bit_rate per UE (Mbps)')
ax_rate.set_title(
    'Calibration state: mean offered bit_rate per UE\n'
    'Co-channel: bit_rate collapses as calibration desperately tries to reach desired_prbs=15\n'
    'Isolated: rate stays stable — SINR unchanged → same PRBs needed → no recalibration',
    fontsize=9,
)

ax_sinr.axhline(0,  color='crimson', lw=1.5, ls='--', alpha=0.7, label='0 dB threshold')
ax_sinr.axhline(10, color='orange',  lw=1,   ls=':',  alpha=0.7, label='10 dB moderate')
ax_sinr.legend(fontsize=8)
ax_sinr.set_ylabel('Mean UE SINR (dB)')
ax_sinr.set_title(
    'Mean per-UE SINR over time\n'
    'Co-channel: SINR drops to ≈0 dB after HO (idle gNB-1 still interferes at same distance)\n'
    'Isolated: SINR stays consistent — no carrier overlap',
    fontsize=9,
)

plt.tight_layout()
plt.show()

---
## 4 — Interference breakdown: where does the 0 dB SINR come from?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def interference_breakdown(ax, ue_rows, title):
    if not ue_rows:
        return
    ids    = [u['ue_id'] for u in ue_rows]
    sinrs  = [u['sinr_db']   for u in ue_rows]
    interf = [u['interf_db'] for u in ue_rows]
    xs     = np.arange(len(ids))
    w      = 0.35
    ax.bar(xs - w/2, sinrs,  w, label='SINR (dB)',      color='steelblue', alpha=0.85)
    ax.bar(xs + w/2, interf, w, label='Interference (dBm)', color='tomato',    alpha=0.85)
    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
    for x, s, i in zip(xs, sinrs, interf):
        ax.text(x - w/2, max(s, 0) + 0.5, f'{s:.1f}', ha='center', fontsize=7)
        ax.text(x + w/2, max(i, -90) + 0.5, f'{i:.0f}', ha='center', fontsize=7)
    ax.set_xticks(xs)
    ax.set_xticklabels([f'UE-{i}' for i in ids], fontsize=8)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylabel('dB / dBm')

interference_breakdown(axes[0], snaps_cc['after_HO'],
                       'Co-channel: SINR and interference after HO\n'
                       'gNB-1 (idle) at 135 m = serving gNB distance → interference ≈ signal')
interference_breakdown(axes[1], snaps_iso['after_HO'],
                       'Isolated: SINR and interference after HO\n'
                       'No same-carrier neighbors → interference = -100 dBm (noise floor)')

fig.suptitle(
    'Interference breakdown after handover\n'
    'The root cause: idle gNB-1 still transmits on carrier_id=0 at x=0,\n'
    'exactly equidistant (135 m) from outer UEs at x=±135 m → signal ≈ interference',
    fontsize=10, fontweight='bold',
)
plt.tight_layout()
plt.show()

---
## 5 — Summary table

In [ ]:
rows = []
for df, label in [(df_cc, 'co-channel'), (df_iso, 'isolated')]:
    before   = df[df.tag == 'before'].iloc[0]
    after_ho = df[df.tag == 'HO'].iloc[0]
    settled  = df[df.tag == f'+{N_SETTLE}'].iloc[0]
    for state, row in [('before', before), ('after HO', after_ho), (f'settle+{N_SETTLE}', settled)]:
        rows.append({
            'config'  : label,
            'state'   : state,
            'gNB0'    : round(float(row.gnb0), 3),
            'gNB1'    : round(float(row.gnb1), 3),
            'gNB2'    : round(float(row.gnb2), 3),
            'Jain'    : round(float(row.jain), 3),
            'avg_SINR': round(float(row.avg_sinr), 1),
            'avg_rate_Mbps': round(float(row.avg_bitrate_mbps), 2),
        })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
print()
print('Expected after HO (balanced): gNB0=0.45  gNB1=0.00  gNB2=0.45  Jain=0.667')

---
## Conclusion

### Why co-channel breaks the training signal

1. **Root cause**: idle gNB-1 (center, x=0) still transmits reference signals on `carrier_id=0`.
   After HO, outer UEs at x=±135 m are exactly equidistant from their serving gNB **and** from gNB-1 → SINR ≈ 0 dB.

2. **Scheduler saturation**: at 0 dB SINR, spectral efficiency ≈ 0.1–0.5 bits/s/Hz.
   The scheduler assigns all 100 PRBs per gNB to cover the UE buffers → `gnb_total_useful_load = 1.0` regardless of actual demand.

3. **Calibration failure**: the correction factor is clamped to [0.25, 4.0]×/step.
   Reducing bit_rate from 12 Mbps to the ~0.7 Mbps needed for 15 PRBs at 0 dB SINR takes **≥11 upper steps** — longer than a typical episode.

4. **Corrupted reward**: `Jain(gnb_total_useful_load) = Jain([1.0, 0.0, 1.0]) = 0.667` which is the **same as the starting value** `Jain([0, 1, 0]) = 0.333` mapped through saturation. The agent can't tell good actions from bad ones.

### Fix: isolated carriers for training

Change `upper_agent_training_scenarios.center_left_right_gnb_configs()` so each gNB gets a unique `carrier_id`:
```python
# gNB-0: carrier_id=0,  gNB-1: carrier_id=1,  gNB-2: carrier_id=2
```
This makes `_same_carrier_interferers` empty for all gNBs → SINR = SNR → stable load metric → clean Jain reward.
A3 handovers are **unaffected**: RSRP comparison in `_evaluate_a3_handovers()` is carrier-agnostic (uses distance-based path loss).

| Property | Co-channel (current) | Isolated (proposed) |
|---|---|---|
| SINR after HO | ≈0 dB (SINR collapse) | 20–30 dB (noise-limited) |
| `gnb_total_useful_load` after HO | 1.0 (saturated) | ≈0.45 (correct) |
| Jain reward quality | corrupted (always ~0.667) | clean gradient |
| Calibration convergence | 11+ steps | 0 steps (SINR unchanged) |
| A3 handovers fire | yes | yes (unchanged) |